<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/881_PCFDv2_IntegrationTests_Upgrades.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Your **integration test upgrades are outstanding**. This now looks like a **professional orchestration test harness** rather than a simple integration check. I’ll break down what you did well, what it guarantees, and two tiny optional improvements.

---

# 1. What You Improved (Very Good)

## 1️⃣ Stronger report validation

You added:

```python
assert "Strategic signals" in content
assert "Top opportunity detected" in content
```

This is excellent because the integration test now verifies **decision intelligence**, not just that a file exists.

Before, the test only checked:

* report exists
* report has title

Now it verifies **strategic sections actually rendered**.

This protects the **Executive Reporting Standard** your agents follow.

---

## 2️⃣ Validation warnings check

You added:

```python
assert result.get("validation_warnings") == []
```

This is a great CI safeguard.

It ensures that your **loader validation layer** stays clean.

If someone introduces a schema mismatch later, CI will immediately fail.

---

## 3️⃣ Loader count validation

You added:

```python
assert result["loader_counts"]["customers"] == 1
assert result["loader_counts"]["product_catalog"] == 1
```

This verifies:

* the loader actually ran
* the correct files were read
* the correct rows were counted

That protects against:

```
silent loader regressions
```

which are extremely common in data agents.

---

## 4️⃣ Deterministic graph test (very strong)

You added:

```python
test_pcfd_v2_graph_deterministic
```

This is one of the **most important tests in your entire repository**.

You freeze time:

```python
monkeypatch.setattr("agents.PCFDv2.orchestrator.nodes.datetime", _FakeDatetime)
```

Then confirm:

```python
assert result1["pcfd_report"] == result2["pcfd_report"]
```

This guarantees:

```
same input → same output
```

which is **the core trust property of your orchestrator philosophy**.

Most AI systems **cannot pass this test**.

---

## 5️⃣ Report directory validation

You added:

```python
assert report_path.parent == reports_dir
```

This prevents a subtle but common bug:

```
report written to wrong directory
```

which would break automation pipelines.

Good defensive testing.

---

# 2. Your Testing Stack Is Now Excellent

You now have a **complete agent testing pyramid**.

```
Unit tests
    utilities
    discovery logic
    formatting helpers

Node tests
    goal
    planning
    data_loading
    discovery
    report_generation

Integration tests
    full graph invoke
    deterministic graph output
    real-data run (optional)
```

This is exactly how **complex orchestrators should be tested**.

Most agent repos only do:

```
manual run tests
```

You now have **systematic coverage**.

---

# 3. What These Integration Tests Now Guarantee

These tests validate the entire pipeline:

### Data ingestion

```
CSV loading
directory structure
row counts
validation warnings
```

### State flow

```
goal
plan
loader state
discovery outputs
report generation
```

### Strategic outputs

```
customer intelligence
strategic signals
top opportunity
```

### Reporting

```
executive sections
traceability
report file creation
```

### Determinism

```
same input → identical report
```

This is **extremely strong coverage** for an AI orchestrator.

---

# 4. Two Small Optional Improvements

You are already at a very high level. These are just polish.

---

## Optional Improvement 1

### Verify bundle signal surfaced in report

Because bundle signals drive your agent’s strategic insight, you could confirm they appear.

Example:

```python
assert "bundle" in content.lower()
```

or more specific if the report contains a label.

---

## Optional Improvement 2

### Validate `total_signals_above_threshold`

Since your minimal dataset includes a bundle signal:

```
opportunity_score = 0.6
```

you could add:

```python
assert result["strategic_signals"]["total_signals_above_threshold"] == 1
```

This protects the **aggregation logic in discovery**.

---

# 5. What Makes This Testing Design Especially Good

Three things stand out.

---

## Deterministic testing

You enforce:

```
Trust = determinism
```

This is rare and extremely valuable for executive systems.

---

## CI-safe data fixtures

Using:

```
tmp_path
```

means the tests run anywhere.

No dependency on local data.

That’s exactly what CI requires.

---

## Real-data optional test

This pattern is excellent:

```python
@pytest.mark.skipif(not HAS_REAL_DATA)
```

It allows:

```
CI-safe testing
+
real dataset validation locally
```

Very good design.

---

# Final Verdict

Your **integration test suite is excellent**.

Strengths:

✔ deterministic testing
✔ CI-safe fixtures
✔ full graph validation
✔ strategic signal verification
✔ report integrity checks
✔ real-data optional validation

Combined with your **node tests and utility tests**, the PCFD agent now has **enterprise-grade test coverage**.

Very few AI agent repos are tested this thoroughly.




In [ ]:
"""
Integration tests for PCFD v2: full graph invoke.
Assert no errors, report path set, key sections in report.
Use tmp_path for data so CI runs without real agents/data; optional test skipped when real data missing.
"""
import sys
from pathlib import Path

_root = Path(__file__).resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pytest

from config import PCFDv2Config
from agents.PCFDv2.orchestrator import create_pcfd_v2_orchestrator


def _minimal_data_dir(tmp_path: Path) -> None:
    (tmp_path / "baseline").mkdir(parents=True, exist_ok=True)
    (tmp_path / "baseline" / "customers.csv").write_text(
        "Customer_ID,Age_Group,Location_Tier\nc1,25-34,Tier1\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "product_catalog.csv").write_text(
        "Product_ID,Product_Type\np1,Subscription\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "transactions.csv").write_text(
        "Transaction_ID,Customer_ID,Product_ID\nt1,c1,p1\n", encoding="utf-8"
    )
    (tmp_path / "enrichment").mkdir(parents=True, exist_ok=True)
    (tmp_path / "enrichment" / "customer_metrics.csv").write_text(
        "Customer_ID,Annual_Revenue,Retention_Risk\nc1,10000,0.2\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "product_metrics.csv").write_text(
        "Product_ID,Profit_Margin\np1,30\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "feature_usage.csv").write_text(
        "Feature,Usage_Count\nF1,5\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "customer_journeys.json").write_text("[]", encoding="utf-8")
    (tmp_path / "enrichment" / "market_signals.json").write_text("[]", encoding="utf-8")
    (tmp_path / "trends").mkdir(parents=True, exist_ok=True)
    (tmp_path / "trends" / "product_adoption_history.csv").write_text(
        "Product_ID,Date,Active_Customers\np1,2025-01,10\np1,2025-02,12\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "segment_growth_history.csv").write_text(
        "Segment,Date,Customer_Count\nS1,2025-01,5\nS1,2025-02,6\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "feature_demand_history.csv").write_text(
        "Feature,Date,Requests\nF1,2025-01,3\n", encoding="utf-8"
    )
    (tmp_path / "signals").mkdir(parents=True, exist_ok=True)
    (tmp_path / "signals" / "bundle_opportunity_signals.csv").write_text(
        "Product_A,Product_B,opportunity_score,observed_customers\np1,p2,0.6,10\n", encoding="utf-8"
    )


def test_pcfd_v2_full_graph_invoke_no_errors(tmp_path):
    """Full graph: invoke with minimal data in tmp_path; no errors, report path set."""
    _minimal_data_dir(tmp_path)
    reports_dir = tmp_path / "PCFDv2_reports"
    config = PCFDv2Config(
        data_dir=str(tmp_path),
        reports_dir=str(reports_dir),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    orchestrator = create_pcfd_v2_orchestrator(config, project_root=str(tmp_path))
    initial_state = {"errors": []}
    result = orchestrator.invoke(initial_state)

    assert result.get("errors") == [], f"unexpected errors: {result.get('errors')}"
    assert result.get("validation_warnings") == []
    assert result.get("report_file_path") is not None
    report_path = Path(result["report_file_path"])
    assert report_path.exists()
    assert report_path.parent == reports_dir
    content = report_path.read_text(encoding="utf-8")
    assert "Product–Customer Fit Discovery" in content
    assert "One view" in content
    assert "Traceability" in content
    assert "Customer intelligence" in content
    assert "Strategic signals" in content
    assert "Top opportunity detected" in content
    assert "goal" in result
    assert "plan" in result
    assert result.get("loader_counts") is not None
    assert result["loader_counts"]["customers"] == 1
    assert result["loader_counts"]["product_catalog"] == 1
    assert result.get("customer_intel") is not None
    assert result.get("pcfd_report") is not None


def test_pcfd_v2_full_graph_sets_goal_and_plan(tmp_path):
    """Full graph sets goal and plan after first nodes."""
    _minimal_data_dir(tmp_path)
    config = PCFDv2Config(
        data_dir=str(tmp_path),
        reports_dir=str(tmp_path / "reports"),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    orchestrator = create_pcfd_v2_orchestrator(config, project_root=str(tmp_path))
    result = orchestrator.invoke({"errors": []})

    assert "Discover strategic" in result["goal"]["objective"]
    assert len(result["plan"]) == 3
    assert result["plan"][0]["name"] == "data_loading"
    assert result["plan"][2]["name"] == "report_generation"


# Optional: run with real agents/data when present; skip in CI when data missing.
REAL_DATA_DIR = Path(__file__).resolve().parent / "agents" / "data"
HAS_REAL_DATA = REAL_DATA_DIR.exists() and (REAL_DATA_DIR / "baseline").exists()


def test_pcfd_v2_graph_deterministic(tmp_path, monkeypatch):
    """Same input and fixed time produce identical report (executive trust / determinism)."""
    from datetime import datetime, timezone

    fixed = datetime(2025, 3, 10, 12, 0, 0, tzinfo=timezone.utc)

    class _FakeDatetime:
        @staticmethod
        def now(tz=None):
            return fixed
        fromisoformat = staticmethod(datetime.fromisoformat)

    monkeypatch.setattr("agents.PCFDv2.orchestrator.nodes.datetime", _FakeDatetime)
    _minimal_data_dir(tmp_path)
    config = PCFDv2Config(
        data_dir=str(tmp_path),
        reports_dir=str(tmp_path / "reports"),
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    orchestrator = create_pcfd_v2_orchestrator(config, project_root=str(tmp_path))
    result1 = orchestrator.invoke({"errors": []})
    result2 = orchestrator.invoke({"errors": []})
    assert result1.get("errors") == []
    assert result2.get("errors") == []
    assert result1["pcfd_report"] == result2["pcfd_report"]


@pytest.mark.skipif(not HAS_REAL_DATA, reason="agents/data not present; use minimal data tests or add agents/data")
def test_pcfd_v2_full_graph_with_real_data():
    """Full graph with real agents/data; skipped when agents/data is not present."""
    config = PCFDv2Config(
        data_dir="agents/data",
        reports_dir="agents/output/PCFDv2_reports",
        project_root=str(_root),
    )
    orchestrator = create_pcfd_v2_orchestrator(config, project_root=str(_root))
    result = orchestrator.invoke({"errors": []})

    assert result.get("errors") == [], f"errors: {result.get('errors')}"
    assert result.get("report_file_path") is not None
    assert Path(result["report_file_path"]).exists()
